## TF-IDF sparse linear pipeline with word and char n-grams

In [155]:
# Inlined utilities and imports
import os
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split
from nltk.stem import SnowballStemmer

# --- preprocessing utilities---
ABBREVIATIONS = {
    'HTA': 'hipertension arterial',
    'IAM': 'infarto agudo miocardio',
    'ACV': 'accidente cerebrovascular',
    'FA': 'fibrilacion auricular',
    'IC': 'insuficiencia cardiaca',
    'ICC': 'insuficiencia cardiaca congestiva',
    'TEP': 'tromboembolismo pulmonar',
    'TVP': 'trombosis venosa profunda',
    'SCA': 'sindrome coronario agudo',
    'IRC': 'insuficiencia renal cronica',
    'IRA': 'insuficiencia renal aguda',
    'RAO': 'retencion aguda orina',
    'EPOC': 'enfermedad pulmonar obstructiva cronica',
    'SAHS': 'sindrome apnea hipopnea sueno',
    'SAOS': 'sindrome apnea obstructiva sueno',
    'CA': 'cancer',
    'NEO': 'neoplasia',
    'LMA': 'leucemia mieloide aguda',
    'LLC': 'leucemia linfocitica cronica',
    'LNH': 'linfoma no hodgkin',
    'GEA': 'gastroenteritis aguda',
    'ERGE': 'enfermedad reflujo gastroesofagico',
    'HDA': 'hemorragia digestiva alta',
    'HDB': 'hemorragia digestiva baja',
    'VHB': 'virus hepatitis B',
    'VHC': 'virus hepatitis C',
    'VHA': 'virus hepatitis A',
    'TCE': 'traumatismo craneoencefalico',
    'TIA': 'accidente isquemico transitorio',
    'AIT': 'accidente isquemico transitorio',
    'DM': 'diabetes mellitus',
    'DM2': 'diabetes mellitus tipo 2',
    'DM1': 'diabetes mellitus tipo 1',
    'DMG': 'diabetes mellitus gestacional',
    'HLD': 'hiperlipidemia',
    'DLP': 'dislipemia',
    'IQ': 'intervencion quirurgica',
    'RN': 'recien nacido',
    'RNPT': 'recien nacido prematuro',
    'RPM': 'rotura prematura membranas',
    'ITU': 'infeccion tracto urinario',
    'PCR': 'parada cardiorrespiratoria',
    'VIH': 'virus inmunodeficiencia humana',
    'CX': 'cirugia',
    'RX': 'radiografia',
    'TAC': 'tomografia axial computarizada',
    'RM': 'resonancia magnetica',
    'SF': 'sindrome febril',
    'SD': 'sindrome',
    'ANTI-D': 'inmunoglobulina anti D',
}

CATALAN_TO_SPANISH = {
    'migranya': 'migranya',
    'cos estrany': 'cuerpo extranos',
    'cos': 'cuerpo',
    'estrany': 'extranos',
    'part': 'parto',
    'malaltia': 'enfermedad',
    'infeccio': 'infeccion',
    'embaras': 'embarazo',
    'diabetis': 'diabetes',
    'hipertensio': 'hipertension',
    'pneumonia': 'neumonia',
    'limfoma': 'linfoma',
    'cataracta': 'catarata',
    'artrosi': 'artrosis',
    'osteoporosi': 'osteoporosis',
    'calcul': 'calculo',
    'dret': 'derecho',
    'dreta': 'derecha',
    'esquerra': 'izquierda',
    'agut': 'agudo',
    'aguda': 'aguda',
    'cronic': 'cronico',
    'cronica': 'cronica',
    'amniodrenatge': 'amniodrenaje',
    'laceracio': 'laceracion',
    'laceracions': 'laceraciones',
    'postpart': 'postparto',
    'cesaria': 'cesarea',
}

ICD10_FULL = re.compile(r'^[A-Z]\d{2}\.?\d*[A-Z0-9]*$')
ICD10_INLINE = re.compile(r'\b([A-Z]\d{2}\.?\d*[A-Z0-9]*)\b')
stemmer = SnowballStemmer('spanish')


def remove_accents(text):
    return unicodedata.normalize(
        'NFC',
        ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    )


def apply_catalan_dict(text):
    for cat, esp in sorted(CATALAN_TO_SPANISH.items(), key=lambda x: -len(x[0])):
        text = re.sub(r'\b' + re.escape(cat) + r'\b', esp, text, flags=re.IGNORECASE)
    return text


def apply_abbreviations(text):
    return ' '.join(ABBREVIATIONS.get(tok.upper(), tok) for tok in text.split())


def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.strip().lower()
    text = remove_accents(text)
    text = apply_catalan_dict(text)
    text = apply_abbreviations(text)
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = [stemmer.stem(tok) for tok in text.split()]
    return re.sub(r'\s+', ' ', ' '.join(tokens)).strip()


def extract_icd_from_literal(literal):
    if not isinstance(literal, str):
        return None
    text = literal.strip().upper()
    if ICD10_FULL.match(text):
        return text.replace('.', '')
    match = ICD10_INLINE.search(text)
    if match:
        return match.group(1).replace('.', '')
    return None


# --- utility functions ---
def load_data(project_root='.'):
    train_raw = pd.read_csv(f"{project_root}/data/raw/training_codification_data.csv")
    icd_raw = pd.read_csv(f"{project_root}/data/raw/icd_d_p_pairs.csv")
    test_raw = pd.read_csv(f"{project_root}/data/raw/test_leaderboard_data.csv")

    codiesp = None
    for cand in [
        f'{project_root}/data/codiesp/final_dataset_v4_to_publish',
        '/kaggle/input/codiesp/final_dataset_v4_to_publish',
        '/kaggle/input/codiesp',
    ]:
        if os.path.isfile(f'{cand}/train/trainX.tsv'):
            cols = ['articleID', 'label', 'Code', 'Literal', 'pos']
            codiesp = pd.concat([
                pd.read_csv(f'{cand}/train/trainX.tsv', sep='\t', header=None, names=cols),
                pd.read_csv(f'{cand}/dev/devX.tsv', sep='\t', header=None, names=cols),
                pd.read_csv(f'{cand}/test/testX.tsv', sep='\t', header=None, names=cols),
            ], ignore_index=True)
            codiesp['Code'] = codiesp['Code'].astype(str).str.upper().str.replace('.', '', regex=False)
            codiesp = codiesp[['Literal', 'Code']].drop_duplicates().reset_index(drop=True)
            break

    return train_raw, icd_raw, test_raw, codiesp


def build_vectorizer(
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 6),
    word_max_features=10000,
    char_max_features=5000,
    min_df=1,
    max_df=1.0,
):
    word_vec = TfidfVectorizer(
        ngram_range=word_ngram_range,
        max_features=word_max_features,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=True,
        preprocessor=None,
        tokenizer=str.split,
        lowercase=False,
        token_pattern=None,
        dtype=np.float32,
    )
    char_vec = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=char_ngram_range,
        max_features=char_max_features,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=True,
        lowercase=False,
        dtype=np.float32,
    )
    return FeatureUnion([('word', word_vec), ('char', char_vec)])


def train_logistic(
    X_train_texts,
    y_train,
    vectorizer=None,
    word_ngram_range=(1, 2),
    char_ngram_range=(3, 6),
    word_max_features=10000,
    char_max_features=5000,
    min_df=1,
    max_df=1.0,
):
    if vectorizer is None:
        vectorizer = build_vectorizer(
            word_ngram_range=word_ngram_range,
            char_ngram_range=char_ngram_range,
            word_max_features=word_max_features,
            char_max_features=char_max_features,
            min_df=min_df,
            max_df=max_df,
        )
    X_train_vec = vectorizer.fit_transform(X_train_texts.astype(str))
    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train_vec, y_train)
    return model, vectorizer


def predict_model(model, vectorizer, X_texts):
    X_vec = vectorizer.transform(X_texts.astype(str))
    return model.predict(X_vec)


def create_submission_from_model(raw_test_df, y_model_pred, out_path="submission.csv"):
    y_category = []
    for v in y_model_pred:
        s = str(v) if not pd.isna(v) else ''
        if not s:
            y_category.append('UNK')
        else:
            m = re.search(r'([A-Za-z0-9])', s)
            y_category.append(m.group(1) if m else s[0])
    submission = pd.DataFrame({
        'id': raw_test_df['id'],
        'Literal': raw_test_df['Literal'],
        'y_category': y_category
    })
    submission.to_csv(out_path, index=False)
    return submission

In [156]:
# Load data
train_raw, icd_raw, test_raw, codiesp = load_data('.')
train_raw.head()

,Code,Literal
0,J9809,Hiperreactividad bronquial
1,J9801,broncoespástica
2,I420,miocardiopatía dilatada
3,Y831,HTA irc 6
4,R5600,Crisis febriles atípicas


In [157]:
# Brief KB preview
icd_raw.head()

,Code,D_P,Description
0,A00,D,Cólera
1,A000,D,"Cólera debido a Vibrio cholerae 01, biotipo ch..."
2,A001,D,"Cólera debido a Vibrio cholerae 01, biotipo El..."
3,A009,D,"Cólera, no especificado"
4,A01,D,Fiebres tifoidea y paratifoidea


In [158]:
# Prepare training and evaluation sets using richer preprocessing and augmented corpora
train_raw['processed'] = train_raw['Literal'].fillna('').apply(preprocess_text)
train_raw['label_first'] = train_raw['Code'].astype(str).str.extract(r'([A-Za-z0-9])', expand=False).fillna('UNK')

icd_raw['Code'] = icd_raw['Code'].astype(str).str.upper().str.replace('.', '', regex=False)
icd_raw['processed'] = icd_raw['Description'].fillna('').apply(preprocess_text)
icd_raw['label_first'] = icd_raw['Code'].astype(str).str.extract(r'([A-Za-z0-9])', expand=False).fillna('UNK')
icd_raw = icd_raw[icd_raw['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

if codiesp is not None:
    codiesp['processed'] = codiesp['Literal'].fillna('').apply(preprocess_text)
    codiesp['label_first'] = codiesp['Code'].astype(str).str.extract(r'([A-Za-z0-9])', expand=False).fillna('UNK')
    codiesp = codiesp[codiesp['processed'] != ''].dropna(subset=['Code']).reset_index(drop=True)

# Test preprocessing includes the ICD shortcut extraction used in the v2 notebook
test_raw['processed'] = test_raw['Literal'].fillna('').apply(preprocess_text)
test_raw['icd_shortcut'] = test_raw['Literal'].apply(extract_icd_from_literal)

stacked_frames = [train_raw[['processed', 'label_first']].rename(columns={'label_first': 'label'})]
stacked_frames.append(icd_raw[['processed', 'label_first']].rename(columns={'label_first': 'label'}))
if codiesp is not None:
    stacked_frames.append(codiesp[['processed', 'label_first']].rename(columns={'label_first': 'label'}))

train_stack = pd.concat(stacked_frames, ignore_index=True)
train_stack = train_stack[(train_stack['processed'] != '') & (train_stack['label'] != 'UNK')].reset_index(drop=True)

X_all = train_stack['processed'].astype(str)
y_all = train_stack['label'].astype(str)

# Train/validation split on the reduced-label task
X_train_texts, X_val_texts, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
X_unlabeled_texts = test_raw['processed'].astype(str)

print('train rows:', len(train_raw), '| icd rows:', len(icd_raw), '| codiesp rows:', 0 if codiesp is None else len(codiesp))
print('test icd_shortcut hits:', test_raw['icd_shortcut'].notna().sum(), '/', len(test_raw))
train_raw[['Literal', 'processed', 'Code']].head()

train rows: 13700 | icd rows: 179742 | codiesp rows: 0
test icd_shortcut hits: 82 / 6667


,Literal,processed,Code
0,Hiperreactividad bronquial,hiperreact bronquial,J9809
1,broncoespástica,broncoespast,J9801
2,miocardiopatía dilatada,miocardiopati dilat,I420
3,HTA irc 6,hipertension arterial insuficient renal cronic 6,Y831
4,Crisis febriles atípicas,crisis febril atip,R5600


In [159]:
from sklearn.metrics import accuracy_score, classification_report

# Fast TF-IDF tuning grid for the sparse linear models
# Uses the already preprocessed text, so this stays lightweight.
tfidf_grid = [
    {
        'word_ngram_range': (1, 2),
        'char_ngram_range': (3, 5),
        'word_max_features': 8000,
        'char_max_features': 3000,
        'min_df': 2,
        'max_df': 0.95,
    },
    {
        'word_ngram_range': (1, 2),
        'char_ngram_range': (2, 5),
        'word_max_features': 12000,
        'char_max_features': 5000,
        'min_df': 2,
        'max_df': 0.90,
    },
]

best_model = None
best_vectorizer = None
best_name = None
best_val_accuracy = -1.0
best_y_val_pred = None

for idx, params in enumerate(tfidf_grid, start=1):
    print(f'\n=== TF-IDF config {idx} ===')
    print(params)
    vectorizer = build_vectorizer(**params)
    X_train_vec = vectorizer.fit_transform(X_train_texts)
    X_val_vec = vectorizer.transform(X_val_texts)

    logistic_model = LogisticRegression(max_iter=800, class_weight='balanced')
    logistic_model.fit(X_train_vec, y_train)
    y_val_pred_log = logistic_model.predict(X_val_vec)
    log_accuracy = accuracy_score(y_val, y_val_pred_log)
    print('Logistic Validation accuracy:', log_accuracy)

    sgd_model = SGDClassifier(loss='hinge', alpha=1e-5, max_iter=1500, random_state=42, class_weight='balanced')
    sgd_model.fit(X_train_vec, y_train)
    y_val_pred_sgd = sgd_model.predict(X_val_vec)
    sgd_accuracy = accuracy_score(y_val, y_val_pred_sgd)
    print('SGD hinge Validation accuracy:', sgd_accuracy)

    if log_accuracy >= sgd_accuracy:
        candidate_model = logistic_model
        candidate_name = 'LogisticRegression'
        candidate_accuracy = log_accuracy
        candidate_pred = y_val_pred_log
    else:
        candidate_model = sgd_model
        candidate_name = 'SGDClassifier(hinge)'
        candidate_accuracy = sgd_accuracy
        candidate_pred = y_val_pred_sgd

    print('Best model for this config:', candidate_name, 'validation accuracy:', candidate_accuracy)

    if candidate_accuracy > best_val_accuracy:
        best_model = candidate_model
        best_vectorizer = vectorizer
        best_name = candidate_name
        best_val_accuracy = candidate_accuracy
        best_y_val_pred = candidate_pred

print('\n=== Best overall sparse linear model ===')
print('Model:', best_name)
print('Validation accuracy:', best_val_accuracy)
print(classification_report(y_val, best_y_val_pred))

full_vectorizer = best_vectorizer  # TF-IDF FeatureUnion selected from tuning grid


=== TF-IDF config 1 ===
{'word_ngram_range': (1, 2), 'char_ngram_range': (3, 5), 'word_max_features': 8000, 'char_max_features': 3000, 'min_df': 2, 'max_df': 0.95}
Logistic Validation accuracy: 0.9441701775698519
SGD hinge Validation accuracy: 0.9494946884127271
Best model for this config: SGDClassifier(hinge) validation accuracy: 0.9494946884127271

=== TF-IDF config 2 ===
{'word_ngram_range': (1, 2), 'char_ngram_range': (2, 5), 'word_max_features': 12000, 'char_max_features': 5000, 'min_df': 2, 'max_df': 0.9}
Logistic Validation accuracy: 0.9476336943317222
SGD hinge Validation accuracy: 0.9527772751944997
Best model for this config: SGDClassifier(hinge) validation accuracy: 0.9527772751944997

=== Best overall sparse linear model ===
Model: SGDClassifier(hinge)
Validation accuracy: 0.9527772751944997
              precision    recall  f1-score   support

           0       1.00      0.98      0.99     13938
           1       0.66      0.82      0.74       108
           2       0.

In [161]:
# Predict on the leaderboard test set using the best sparse linear model pipeline
# Transform test texts via the tuned TF-IDF vectorizer
X_unlabeled_tfidf = full_vectorizer.transform(X_unlabeled_texts)

# Ensure predictions are 1D
y_model_log = best_model.predict(X_unlabeled_tfidf)
y_model_log = np.asarray(y_model_log).ravel()

# If the literal contains an explicit ICD code, use its first character as a high-confidence shortcut
shortcut_mask = test_raw['icd_shortcut'].notna()
if shortcut_mask.any():
    shortcut_preds = test_raw.loc[shortcut_mask, 'icd_shortcut'].astype(str).str[0].values
    y_model_log[shortcut_mask.to_numpy()] = shortcut_preds

# Build df_compare for inspection
y_model_series = pd.Series(y_model_log, index=test_raw.index)
df_compare = pd.DataFrame({
    'Literal': test_raw['Literal'],
    'Predicted Code': y_model_series
})
# Create final submission in leaderboard format (id, Literal, y_category) using raw test file and tuned TF-IDF predictions
submission = create_submission_from_model(test_raw, y_model_log, out_path='submission.csv')
submission.head()

,id,Literal,y_category
0,1,AMNIODRENAJE,7
1,2,Hiperparatiroidismo primario,E
2,3,MIGRANYA parto,G
3,4,VHC,B
4,5,Absceso mama izq,N
